# 🧬 18 — Likelihoods, priors, and hidden structural states

**Audience:** biology student moving toward research · **Time:** ~4 hours · **Hardware:** free Colab CPU · **Framework:** NumPy · **Prerequisites:** 08, 10, 12, 17; logarithms and basic probability

**Goals:** explain the concept, run and modify the toy model, identify one failure mode, and connect the method to an original paper. All data are synthetic and for research and education.


## 1. Biological and computational motivation

Two conformations can be mixed in one experiment. Averaging them together may blur a mobile domain. A probabilistic model treats class and pose as hidden variables and asks how likely each image would be under each candidate map. This provides soft assignments rather than pretending every noisy particle has an obvious label. A prior can stabilize weak data, but it can also pull the answer toward an expected structure.


## 2. Theory and assumptions

For candidate map `V`, image `y`, noise scale `σ`, and forward operator `F`, a Gaussian toy likelihood is `p(y|V) ∝ exp(-||y−F(V)||²/(2σ²))`. Bayes' rule gives `p(V|y) ∝ p(y|V)p(V)`. Expectation–maximization alternates: estimate probabilities of hidden classes/poses (E-step), then update model parameters using those probabilities (M-step). Real cryo-EM likelihoods must include CTF, dose, rotations, and nontrivial noise.


## 3. Runnable experiment

Predict a result before running. In Colab, NumPy is included; elsewhere install it with `%pip install numpy` if needed.


In [ ]:
import numpy as np
SEED=7
print("NumPy",np.__version__,"| device: CPU | seed:",SEED)


In [ ]:
rng=np.random.default_rng(SEED)
data=np.r_[rng.normal(-1,.4,80),rng.normal(1,.5,120)]
means=np.array([-.5,.5]); weights=np.array([.5,.5]); variance=.25
for iteration in range(30):
    like=np.exp(-.5*(data[:,None]-means[None,:])**2/variance)*weights
    responsibility=like/like.sum(axis=1,keepdims=True)
    weights=responsibility.mean(axis=0)
    means=(responsibility*data[:,None]).sum(0)/responsibility.sum(0)
print('class means',means.round(2),'class weights',weights.round(2))
print('ambiguous particles',int(np.sum((responsibility[:,0]>.4)&(responsibility[:,0]<.6))))
assert means[0]<-.5 and means[1]>.5


## 4. Interpretation and challenge

The algorithm knew there were two classes and used one scalar feature. Real class count and feature representation are unknown. Multiple initializations, independent half-sets, and biological tests are needed before a class becomes a conformation claim.

**Change one thing:** Start both means at zero. What happens? Fit three classes to the same data. Describe one metric and one visual control that could detect overclassification in actual subtomograms.


## 5. Paper bridge

Compare the objective in [fast alignment plus maximum likelihood](https://doi.org/10.1093/bioinformatics/bty267) with the biological-state question in the [2025 content–transformation study](https://xulabs.github.io/publication/).

Record the paper’s input, output, supervision, split unit, central assumption, strongest comparison, and one alternate explanation.
